In [2]:
import os
import tifffile
import rasterio

import cv2
import numpy as np

import leafmap.leafmap as leafmap
#from samgeo import SamGeo2

import geopandas as gpd
import pickle
from pyproj import Transformer

from utils.raster_tools import Raster_profile #class type for raster profile
import matplotlib.pyplot as plt

### See the overall

In [ ]:
clipped_theos_file = "theos/clipped_IMG_T2V_20250119034323_ORTHO_PMS_32_small.tif"

# m = leafmap.Map(center=[(lat_start + lat_end)/2, (long_start + long_end)/2], zoom=16, height="800px")
m = leafmap.Map(center=[12.908807, 100.922147], zoom=16 , height="800px") 

m.add_basemap("Satellite") 
m.add_raster(clipped_theos_file, layer_name="Theos") 
m

## Read the cliped Google image

In [ ]:
theos_Profile = Raster_profile(clipped_theos_file) 

### Specify the grid's spacing

In [ ]:
from pyproj import Transformer
import csv

def setup_polygon(long_start, lat_start, long_end, lat_end, crs_source="EPSG:4326", crs_target="EPSG:32647"): 
    bbox = [long_start, lat_start, long_end, lat_end] 
    coordinates = [
        [bbox[0], bbox[3]],  # Top-left corner (min_lon, max_lat)
        [bbox[2], bbox[3]],  # Top-right corner (max_lon, max_lat)
        [bbox[2], bbox[1]],  # Bottom-right corner (max_lon, min_lat)
        [bbox[0], bbox[1]],  # Bottom-left corner (min_lon, min_lat)
        [bbox[0], bbox[3]]   # Closing the polygon by repeating the first point
        ]

    transformer = Transformer.from_crs(crs_source, crs_target, always_xy=True)
    poly_gons = []
    for coord in coordinates: 
        easting, northing = transformer.transform(coord[0], coord[1])
        poly_gons.append([easting, northing])

    return poly_gons, bbox, coordinates

def save_stats(stats, path_npz, path_csv):
    np.savez(path_npz, **stats)

    with open(path_csv, 'w', newline='') as file:
        writer = csv.writer(file)
        # Write header
        writer.writerow(['Key', 'Value'])
        # Write data line by line
        for key, value in stats.items():
            writer.writerow([key, value])
            
def read_npz(npz_filename):
    read_stats = dict(np.load(npz_filename))

    read_dict = {}
    for key, value in read_stats.items():
        try:
            read_dict[key] = value.item() 
        except:
            read_dict[key] = value

    return read_dict

In [ ]:
long_start, lat_start = theos_Profile.get_longlat_from_image_pixels(0, 0, crs_dst="EPSG:4326")

In [ ]:
long_end, lat_end = theos_Profile.get_longlat_from_image_pixels(7001, 7053, crs_dst="EPSG:4326")

In [ ]:
zoom_level = 18 # Google resolution // the higher zoom level >> higher resolution 
lat_diff   = np.abs(lat_end - lat_start)/10
long_diff  = lat_diff
crs_source = "EPSG:4326" # Google 
crs_target = "EPSG:32647" # Theos

# EDIT HERE 

In [ ]:
slice_row   = 1
slice_column = 5

slices_path = "ISP0704-Zoom%d" % zoom_level


slice_subpath  = os.path.join(slices_path, "%000d-%000d" % (slice_row, slice_column))
slice_google_filename = os.path.join(slice_subpath, "google.tif") 
warped_slice_google_filename = os.path.join(slice_subpath, "warped_google.tif") 
slice_theos_filename  = os.path.join(slice_subpath, "theos.tif") 
npz_filename  = os.path.join(slice_subpath, "stats.npz") 
csv_filename = os.path.join(slice_subpath, "stats.csv") 

os.makedirs(slices_path, exist_ok=True)
os.makedirs(slice_subpath, exist_ok=True)


long_start_temp = long_start  + (slice_column-1)*long_diff.item()  
# lat_start_temp  = lat_start  - (slice_no)*lat_diff.item()
lat_start_temp  = lat_start - (slice_row-1)*lat_diff.item()

long_end_temp = long_start + (slice_column)*long_diff.item()  
#lat_end_temp  = lat_start  - (slice_no+1)*lat_diff.item()
lat_end_temp  = lat_start  -  (slice_row)*lat_diff.item()


print("Slice no. %000d-%000d" % (slice_row, slice_column)) 
print("Start: LON: %f LAT: %f" % (long_start_temp, lat_start_temp)) 
print("End  : LON: %f LAT: %f" % (long_end_temp, lat_end_temp)) 

poly_gons, bbox, coordinates = setup_polygon(long_start_temp, lat_start_temp, long_end_temp, lat_end_temp, crs_source=crs_source, crs_target=crs_target)

leafmap.map_tiles_to_geotiff(output=slice_google_filename, bbox=bbox, zoom=zoom_level, source="Satellite", overwrite=True)
leafmap.clip_image(clipped_theos_file, poly_gons, slice_theos_filename)


stats = {"slice_row": slice_row, 
        "slice_column": slice_column,
        "slice_theos_filename": slice_theos_filename,
        "slice_google_filename": slice_google_filename,  
        "long_start_temp": long_start_temp,
        "lat_start_temp": lat_start_temp,
        "long_end_temp": long_end_temp,
        "lat_end_temp": lat_end_temp,
        "poly_gons": poly_gons,
        "bbox":bbox,
        "coordinates": coordinates,
        "crs_source": crs_source,
        "crs_target": crs_target,
        "long_diff": long_diff,
        "lat_diff": lat_diff}

save_stats(stats, npz_filename, csv_filename)

## Test sliced data

### Stats reading

In [ ]:
read_dict = read_npz(npz_filename)
read_dict

### Map reading

In [ ]:
m = leafmap.Map()    

slice_theos_filename_prev = 'ISP0704-Zoom18/5-1/theos.tif'
slice_google_filename_prev = 'ISP0704-Zoom18/5-1/google.tif'
m.add_raster(slice_google_filename_prev, layer_name="Google-prev")
m.add_raster(slice_google_filename, layer_name="Google")
m.add_raster(slice_theos_filename_prev, layer_name="Theos-prev") 

m.add_raster(slice_theos_filename, layer_name="Theos") 
m

### Test warping

In [ ]:
from plantcv import plantcv as pcv
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import numpy as np
import tifffile

from utils.tools import get_raster_data, image_enhancement, save_raster_and_write_meta

In [ ]:
# slice_no   = 0

slices_path     = "ISP0704-Zoom%d" % zoom_level
slice_subpath  = os.path.join(slices_path, "%000d-%000d" % (slice_row, slice_column))

slice_google_filename = os.path.join(slice_subpath, "google.tif") 
warped_slice_google_filename = os.path.join(slice_subpath, "warped_google.tif") 
slice_theos_filename  = os.path.join(slice_subpath, "theos.tif")  
path_warp_npz  = os.path.join(slice_subpath, "warp_stats.npz")  
path_warp_csv  = os.path.join(slice_subpath, "warp_stats.csv")   


In [ ]:
imgA, _ = get_raster_data(slice_google_filename) 
imgA = np.transpose(imgA, (1, 2, 0))  # Convert from (bands, height, width) to (height, width, bands)
 
imgB, _  = get_raster_data(slice_theos_filename) 
imgB = np.transpose(imgB, (1, 2, 0))  # Convert from (bands, height, width) to (height, width, bands)
imgB = imgB[:,:,:3]
imgB = image_enhancement(imgB) #แก้ exposure intensity (ไว้เพิ่มใน report)

In [ ]:
from utils.interactive_tools import Find_correspondences

%matplotlib widget
marker_AB = Find_correspondences(imgA, imgB, figsize=(13, 8))

In [ ]:
import cv2

point_src  = np.array(marker_AB.points[0])
point_dst  = np.array(marker_AB.points[1])

Homography, status = cv2.findHomography(point_src, point_dst) 

In [ ]:
target_size = (imgB.shape[1], imgB.shape[0])
im_dst = cv2.warpPerspective(imgA, Homography, target_size) 

In [ ]:
warp_stats = {
    "point_src": point_src,
    "point_dst": point_dst,
    "Homography": Homography,
    "target_size": target_size, 
    "src_img_filename": slice_google_filename,
    "dst_img_filename": slice_theos_filename,
    "result_image_filename": warped_slice_google_filename
}

In [ ]:
save_stats(warp_stats, path_warp_npz, path_warp_csv)

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(10,5)) 
axs[0].imshow(imgA[:,:,0:3])  # Display the first three channels (RGB) of the clipped image
axs[0].set_title('Google') # Set a title for the first subplot 

axs[1].imshow(imgB[:,:,0:3])  # Display the first three channels (RGB) of the clipped satellite image
axs[1].set_title('Theos') # Set a title for the second subplot 

axs[2].imshow(im_dst[:,:,0:3])  # Display the first three channels (RGB) of the clipped satellite image
axs[2].set_title('Google (warped)') # Set a title for the second subplot 
 
fig.tight_layout()

In [ ]:
im_dst_4D = np.zeros((imgB.shape[0], imgB.shape[1], 4)) 
im_dst_4D[:,:,:3] = im_dst 
im_dst_4D[:,:, 3] = 254
im_dst_4D = im_dst_4D.transpose(2, 0, 1)
im_dst_4D = im_dst_4D.astype(np.uint8)

destination_tif = warped_slice_google_filename
meta_source_tif = slice_theos_filename
save_raster_and_write_meta(im_dst_4D , destination_tif, meta_source_tif)

In [ ]:
import leafmap.leafmap as leafmap
m = leafmap.Map()
m.add_raster(slice_google_filename, layer_name="Google") 
m.add_raster(slice_theos_filename, layer_name="theos") 
m.add_raster(warped_slice_google_filename, layer_name="Google (warped)")  
m

### Test SamGeo2

In [ ]:
from samgeo import SamGeo2 

In [ ]:
sam = SamGeo2(
    model_id="sam2-hiera-large", 
    automatic=False
)

In [ ]:
# slice_no   = 0

slices_path     = "ISP0704-Zoom%d" % zoom_level
slice_subpath   = os.path.join(slices_path, "%000d-%000d" % (slice_row, slice_column)) 
warped_slice_google_filename = os.path.join(slice_subpath, "warped_google.tif")  
target_dir      = os.path.join(slice_subpath, "samgeo2mask")  
os.makedirs(target_dir, exist_ok=True)


In [ ]:
print("PLEASE SAVE THE MASK under Folder: %s" % target_dir)

In [ ]:
sam.set_image(warped_slice_google_filename)
sam.show_map()

In [ ]:
mask_filename = os.path.join(slice_subpath,"samgeo2mask", "masks.tif") 

In [ ]:
import leafmap.leafmap as leafmap
m = leafmap.Map() 
m.add_raster(slice_theos_filename, layer_name="theos") 
m.add_raster(warped_slice_google_filename, layer_name="Google (warped)")  
m.add_raster(mask_filename, cmap="jet", layer_name="Mask (warped)")  
m

## Mask improvement

In [ ]:
from utils.mask_tools import Mask_profile

from plantcv import plantcv as pcv
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import numpy as np
import tifffile

from utils.tools import get_raster_data, image_enhancement, save_raster_and_write_meta

In [ ]:
#slice_no =0

mask_filename = os.path.join(slice_subpath, "samgeo2mask", "masks.tif") 
center_geojson = os.path.join(slice_subpath, "samgeo2mask", "masks_fg_markers.geojson")  
warped_slice_google_filename = os.path.join(slice_subpath, "warped_google.tif") 
sat_image, _ = get_raster_data(warped_slice_google_filename)

In [ ]:
Mask_obj = Mask_profile(mask_filename, center_geojson_file=center_geojson, center_geojson_crs="EPSG:4326") 
Mask_obj.show_mask_order(figsize=(10, 5), fontsize=8, alpha=0.75, satellite_image=sat_image)

In [ ]:
sat_image.shape
sat_image = sat_image.transpose(1,2,0)[:,:,:3]

In [ ]:
gray_bf_operation = Mask_obj.mask.copy()

for manual_index in range(Mask_obj.mask.max()):

    
    gray_bf = Mask_obj.get_a_binary_mask(manual_index)

    record_af_list = []
    gray_af, record_af = Mask_obj.filling_holes(1*gray_bf)
    record_af_list.append(record_af)
    gray_af, record_af = Mask_obj.erosion(gray_af.astype(np.uint8), kernel_size=3) # ลองเพิ่ม erosion ดูว่าช่วยลด noise ได้ไหม
    record_af_list.append(record_af)
    gray_af, record_af = Mask_obj.dilation(gray_af.astype(np.uint8), kernel_size=17)
    record_af_list.append(record_af)

    mask_2D = Mask_obj.update_a_binary_mask(gray_af, manual_index, record_af_list) 

    # fig, axs = plt.subplots(1, 2, figsize=(10, 3))

    # axs[0].imshow(sat_image)
    # axs[0].imshow(gray_bf, cmap='Oranges', alpha=0.45)
    # axs[0].set_title("Mask id %d before [top]" % manual_index)
    

    # axs[1].imshow(sat_image)
    # axs[1].imshow(gray_af, cmap='Oranges', alpha=0.45)
    # axs[1].set_title("Mask id %d after [top]" % manual_index) 

    # plt.tight_layout() 

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(10, 12))
axs[0].imshow(sat_image)
axs[0].imshow(gray_bf_operation, cmap='turbo', alpha=0.35) 
axs[0].set_title("before")

axs[1].imshow(sat_image)
axs[1].imshow(mask_2D, cmap='turbo', alpha=0.35) 
axs[1].set_title("after")

In [ ]:
edited_mask_filename = os.path.join(slice_subpath,"samgeo2mask", "masks_edited.tif") 
meta_mask_filename   = os.path.join(slice_subpath, "samgeo2mask", "masks.tif") 
mask_2D              = mask_2D.reshape(1, mask_2D.shape[0], mask_2D.shape[1])
save_raster_and_write_meta(mask_2D, edited_mask_filename, meta_mask_filename)

In [ ]:
m = leafmap.Map()
m.add_raster(warped_slice_google_filename, layer_name="Image") 
m.add_circle_markers_from_xy(center_geojson, radius=3, color="red", fill_color="yellow", fill_opacity=0.8
) 
m.add_raster(mask_filename, cmap="jet", layer_name="Building masks (before)") 
m.add_raster(edited_mask_filename, cmap="jet", layer_name="Building masks (after)") 
m

In [ ]:
bb_filename = os.path.join(slice_subpath, "samgeo2mask", "boundbox.geojson") 
 
gdf = Mask_obj.make_boundboxes()
gdf.to_file(bb_filename, driver='GeoJSON') 

In [ ]:
m = leafmap.Map()
m.add_raster(warped_slice_google_filename, layer_name="Image") 
m.add_circle_markers_from_xy(center_geojson, radius=3, color="red", fill_color="yellow", fill_opacity=0.8
)  
m.add_raster(edited_mask_filename, cmap="jet", layer_name="Building masks (after)")  
m.add_vector(bb_filename, layer_name="Bounding Boxes")
m

# การรวมภาพที่ slice หลายๆภาพเข้าด้วยกัน


## Load Masks

In [ ]:
from google.oauth2 import service_account
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload
import io
import os

SCOPES = ['https://www.googleapis.com/auth/drive.readonly']
SERVICE_ACCOUNT_FILE = 'credentials.json'
ROOT_FOLDER_ID = '1QGFj50j3a19VwgDHTcQein3ReIuPGCcX'
TARGET_NAME = 'masks.tif'   # ชื่อไฟล์ที่ต้องการหาแบบตรงเป๊ะ

creds = service_account.Credentials.from_service_account_file(
    SERVICE_ACCOUNT_FILE, scopes=SCOPES
)
service = build('drive', 'v3', credentials=creds)


def list_children(folder_id):

    all_items = []
    page_token = None

    while True:
        response = service.files().list(
            q=f"'{folder_id}' in parents and trashed = false",
            fields="nextPageToken, files(id, name, mimeType)",
            pageToken=page_token
        ).execute()

        all_items.extend(response.get("files", []))
        page_token = response.get("nextPageToken", None)

        if not page_token:
            break

    return all_items


def find_files_recursive(folder_id, target_name, current_path=""):
    """
    ไล่หาไฟล์ชื่อ target_name จาก folder นี้และทุก subfolder
    current_path ใช้เก็บ path ว่าเจอในโฟลเดอร์ไหน
    """
    matched_files = []
    items = list_children(folder_id)

    for item in items:
        item_name = item["name"]
        item_id = item["id"]
        mime_type = item["mimeType"]

        if mime_type == "application/vnd.google-apps.folder":
            subfolder_path = f"{current_path}/{item_name}" if current_path else item_name
            matched_files.extend(
                find_files_recursive(item_id, target_name, subfolder_path)
            )

        else:
            if item_name == target_name:
                matched_files.append({
                    "id": item_id,
                    "name": item_name,
                    "path": current_path.split('/')[1] if current_path else "/"
                })

    return matched_files


results = find_files_recursive(ROOT_FOLDER_ID, TARGET_NAME)

if not results:
    print("ไม่เจอไฟล์")
else:
    print(f"เจอทั้งหมด {len(results)} ไฟล์\n")
    for i, f in enumerate(results, start=1):
        print(f"{i}. name={f['name']} | id={f['id']} | path={f['path']}")

def download_file(file_id, save_path):
    request = service.files().get_media(fileId=file_id)
    with open(save_path, "wb") as fh:
        downloader = MediaIoBaseDownload(fh, request)
        done = False
        while not done:
            status, done = downloader.next_chunk()
            

os.makedirs("downloads2", exist_ok=True)

for i, f in enumerate(results, start=1):
    save_name = f"{f['path']}_{f['name']}"
    save_path = os.path.join("downloads2", save_name)
    download_file(f["id"], save_path)

#print("โหลดครบแล้ว")

## Load Google

In [3]:
from google.oauth2 import service_account
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload
import io
import os

SCOPES = ['https://www.googleapis.com/auth/drive.readonly']
SERVICE_ACCOUNT_FILE = 'credentials.json'
ROOT_FOLDER_ID = '1QGFj50j3a19VwgDHTcQein3ReIuPGCcX'
TARGET_NAME_LIST = ['masks.tif','google.tif']   # ชื่อไฟล์ที่ต้องการหาแบบตรงเป๊ะ

creds = service_account.Credentials.from_service_account_file(
    SERVICE_ACCOUNT_FILE, scopes=SCOPES
)
service = build('drive', 'v3', credentials=creds)

def list_children(folder_id):
    all_items = []
    page_token = None

    while True:
        response = service.files().list(
            q=f"'{folder_id}' in parents and trashed = false",
            fields="nextPageToken, files(id, name, mimeType)",
            pageToken=page_token
        ).execute()

        all_items.extend(response.get("files", []))
        page_token = response.get("nextPageToken", None)

        if not page_token:
            break

    return all_items


def find_files_recursive(folder_id, target_name, current_path=""):
    matched_files = []
    items = list_children(folder_id)

    for item in items:
        item_name = item["name"]
        item_id = item["id"]
        mime_type = item["mimeType"]

        if mime_type == "application/vnd.google-apps.folder":
            subfolder_path = f"{current_path}/{item_name}" if current_path else item_name
            matched_files.extend(
                find_files_recursive(item_id, target_name, subfolder_path)
            )

        else:
            if item_name == target_name:
                matched_files.append({
                    "id": item_id,
                    "name": item_name,
                    "path": current_path.split('/')[1] if current_path else "/"
                })

    return matched_files

def download_file(file_id, save_path):
    request = service.files().get_media(fileId=file_id)
    with open(save_path, "wb") as fh:
        downloader = MediaIoBaseDownload(fh, request)
        done = False
        while not done:
            status, done = downloader.next_chunk()

for TARGET_NAME in TARGET_NAME_LIST:
    results = find_files_recursive(ROOT_FOLDER_ID, TARGET_NAME)

    if not results:
        print("ไม่เจอไฟล์")
    else:
        print(f"เจอทั้งหมด {len(results)} ไฟล์\n")
        for i, f in enumerate(results, start=1):
            print(f"{i}. name={f['name']} | id={f['id']} | path={f['path']}")

    os.makedirs("downloads2", exist_ok=True)

    for i, f in enumerate(results, start=1):
        # กันชื่อชนกัน เพราะทุกไฟล์ชื่อเดียวกันหมด
        save_name = f"{f['path']}_{f['name']}"
        save_path = os.path.join("downloads2", save_name)
        download_file(f["id"], save_path)

print("โหลดครบแล้ว")

เจอทั้งหมด 26 ไฟล์

1. name=masks.tif | id=11hl4e9vkVxyMe-cZGmgPbIqdNSsVtKhL | path=4-5
2. name=masks.tif | id=16S0qmTkgsqXTibAuorN2kt9apMofVg6R | path=2-5
3. name=masks.tif | id=1_fuiQxREIQ1VkK3Ok3Hx_Ov12ph-Knxt | path=1-5
4. name=masks.tif | id=1fSSWPvjJEEN4AUwEBC-PIXef9MhXvft0 | path=5-5
5. name=masks.tif | id=1Y_uRT1FeEgcHve22iNt6n_KhtS1X_3vt | path=3-5
6. name=masks.tif | id=120uZT3zq3gbTAIYaSPTQQFcdL16UuKrg | path=5-4
7. name=masks.tif | id=1rXFtY0qiDvCNGkSW1CJQadUS91O8JshK | path=1-4
8. name=masks.tif | id=1fiBduoQ7hYyu8CbZJqbYqpOvYhF-tfgj | path=3-4
9. name=masks.tif | id=1ZfUzpYKNdw9RZV2maYdpKENpwTBo9-za | path=2-4
10. name=masks.tif | id=1ZVnDISot3jmRq8paNrcBTJaGui3FbXSR | path=4-4
11. name=masks.tif | id=12AOMhI-ZM51GKhu-orc7Lg50ueQynJFe | path=5-3
12. name=masks.tif | id=1bEvOadU7Y49iaAgK1-vWD9TEgVf84j0I | path=4-3
13. name=masks.tif | id=1_n9-UFmy22Rr2l4ftmv0ddJ3WFsNLbxa | path=3-3
14. name=masks.tif | id=1WRrjEflkS2DSZ4-OW6Y-JiuYOFgDM_ix | path=2-3
15. name=masks.tif | id

## Check 

In [5]:
import os
import glob

filename_google = glob.glob("downloads2/*google.tif")
filename_masks = glob.glob("downloads2/*masks.tif")
m = leafmap.Map()    
for filename in filename_masks:
    slice_theos_filename_prev_test = filename
    m.add_raster(slice_theos_filename_prev_test, layer_name="Theos-prev") 

#for filename in filename_google:
    #slice_google_filename_prev_test = filename
    #m.add_raster(slice_google_filename_prev_test, layer_name="Google (warped)")
m

Map(center=[12.9281735, 100.90079800000001], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_…